# 1장 3강: 데이터 특성에 따른 가설검정 기법 선택 — 실습문제

## 실습 목표

- 비교할 변수의 척도와 집단 관계를 확인할 수 있다.
- Shapiro-Wilk 검정으로 정규성을 확인할 수 있다.
- Levene 검정으로 등분산성을 확인할 수 있다.
- 가정 점검 결과에 따라 독립표본 t검정과 Welch t검정을 선택할 수 있다.
- 선택한 검정의 p-value를 해석하여 데이터에 근거한 결론을 작성할 수 있다.

## 실습 환경 / 데이터

- Python
- pandas
- scipy.stats
- `ames_housing.csv`

| 컬럼 | 의미 | 유형 |
|---|---|---|
| `SalePrice` | 주택 판매가격 | 연속형 |
| `KitchenQual` | 주방 품질 | 범주형·순서형 |
| `HeatingQC` | 난방 품질 | 범주형·순서형 |

> 모든 판단의 유의수준은 `α = 0.05`입니다.  
> 표본 추출에는 `random_state=42`를 사용하여 실행할 때마다 같은 결과가 나오도록 합니다.


## 실습 준비

1. pandas와 `scipy.stats`를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 전체 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.
# 1. 라이브러리 불러오기
import pandas as pd
from scipy import stats

# 2. 데이터 불러오기
df = pd.read_csv("ames_housing.csv")

# 3-1. 데이터 크기 확인 (행, 열)
print("데이터 크기 (행, 열): ", df.shape)

# 3-2. 전체 결측치 수 확인
print("전체 결측치 수: ", df.isnull().sum().sum())

# 3-3. 컬럼명 확인
print("\n컬럼명: ")
print(df.columns.tolist())

# 3-4. 상위 5개 행 확인
print("\n상위 5개 행:")
print(df.head())

데이터 크기 (행, 열):  (1460, 10)
전체 결측치 수:  0

컬럼명: 
['SalePrice', 'GrLivArea', 'LotArea', 'OverallQual', 'KitchenQual', 'CentralAir', 'HeatingQC', 'PavedDrive', 'Neighborhood', 'YearBuilt']

상위 5개 행:
   SalePrice  GrLivArea  LotArea  OverallQual KitchenQual CentralAir  \
0     208500       1710     8450            7          Gd          Y   
1     181500       1262     9600            6          TA          Y   
2     223500       1786    11250            7          Gd          Y   
3     140000       1717     9550            7          Gd          Y   
4     250000       2198    14260            8          Gd          Y   

  HeatingQC PavedDrive Neighborhood  YearBuilt  
0        Ex          Y      CollgCr       2003  
1        Ex          Y      Veenker       1976  
2        Ex          Y      CollgCr       2001  
3        Gd          Y      Crawfor       1915  
4        Ex          Y      NoRidge       2000  


---

## 필수 1. 정규성은 충족하지만 등분산성이 위반된 경우

### 문제 1-1. 주방 품질 `Gd`와 `TA` 집단의 판매가격 비교

#### 문제 설명

주방 품질이 `Gd`(Good)인 주택과 `TA`(Typical/Average)인 주택의 판매가격을 비교하려고 합니다. 각 집단에서 30개씩 표본을 추출한 뒤 정규성과 등분산성을 확인하고 적절한 t검정 방법을 선택하세요.

#### 요구사항

1. `KitchenQual == "Gd"`인 주택의 `SalePrice`에서 30개를 추출해 `group_gd`에 저장하세요.
2. `KitchenQual == "TA"`인 주택의 `SalePrice`에서 30개를 추출해 `group_ta`에 저장하세요.
3. 두 집단의 표본 수와 평균을 확인하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 두 집단에 Levene 등분산 검정을 수행하세요.
6. 각 가정의 p-value를 0.05와 비교하여 충족 여부를 판단하세요.
7. 다음 규칙에 따라 t검정 방법을 선택하세요.
   - 두 집단 모두 정규성 충족 + 등분산성 충족: 독립표본 t검정
   - 두 집단 모두 정규성 충족 + 등분산성 위반: Welch t검정
8. 선택한 검정을 실행하고 검정통계량과 p-value를 출력하세요.
9. 두 집단의 판매가격 차이가 통계적으로 유의한지 해석하세요.

#### 해석 질문

**Q1.** Shapiro-Wilk 검정의 귀무가설은 무엇인가요?  
**Q2.** Levene 검정의 귀무가설은 무엇인가요?  
**Q3.** 가정 점검 결과 어떤 t검정 방법을 선택해야 하나요?  
**Q4.** 선택한 검정의 결과에 따르면 두 집단의 판매가격 차이는 유의한가요?

#### 제출 결과

- 집단별 표본 수와 평균
- 정규성 및 등분산성 검정 결과
- t검정 방법 선택과 선택 근거
- 최종 검정통계량과 p-value
- 결과 해석
- Q1~Q4 답변


In [3]:
# 필수 1 코드를 작성하세요.
import pandas as pd
from scipy import stats

df = pd.read_csv("ames_housing.csv")

# 1. Gd 표본 30개 추출
group_gd = (
    df.loc[df["KitchenQual"]=="Gd", "SalePrice"]
    .dropna()
    .sample(n=30, random_state=42)
)

# 2. TA 표본 30개 추출
group_ta = (
    df.loc[df["KitchenQual"]=="TA", "SalePrice"]
    .dropna()
    .sample(n=30, random_state=42)
)

# 3. 표본 수와 평균 확인
print("Gd 표본 수: ", len(group_gd))
print("TA 표본 수: ", len(group_ta))
print("Gd 평균: ", group_gd.mean())
print("TA 평균: ", group_ta.mean())

# 4. 정규성 검정 (Shapiro-Wilk)
shapiro_gd = stats.shapiro(group_gd)
shapiro_ta = stats.shapiro(group_ta)

print("\n[Gd 정규성]")
print("statistic: ", shapiro_gd.statistic)
print("p-value: ", shapiro_gd.pvalue)

print("\n[TA 정규성]")
print("statistic: ", shapiro_ta.statistic)
print("p-value: ", shapiro_ta.pvalue)

# 5. 등분산성 검정 (Levene)
levene_result = stats.levene(group_gd, group_ta)
print("\n[등분산성]")
print("statistic: ", levene_result.statistic)
print("p-value: ", levene_result.pvalue)

# 6~7. 가정 판단 후 t검정 방법 선택
alpha = 0.05
normal_gd = shapiro_gd.pvalue >= alpha
normal_ta = shapiro_ta.pvalue >= alpha
equal_var = levene_result.pvalue >= alpha

if normal_gd and normal_ta and equal_var:
    test_result = stats.ttest_ind(group_gd, group_ta, equal_var=True)
    method = "독립표본 t검정 (등분산 가정)"
elif normal_gd and normal_ta and not equal_var:
    test_result = stats.ttest_ind(group_gd, group_ta, equal_var=False)
    method = "Welch t검정 (등분산 가정하지 않음)"
else:
    method = "정규성 위반 → 비모수 검정 고려 필요"
    test_result = stats.mannwhitneyu(group_gd, group_ta)

print("\n선택한 검정 방법: ", method)

# 8. 검정통계량과 p-value 출력
print("검정통계량: ", test_result.statistic)
print("p-value: ", test_result.pvalue)

# 9. 결과 해석
if test_result.pvalue < alpha:
    print(f"\np-value({test_result.pvalue:.6f}) < 유의수준({alpha})")
    print("→ 귀무가설 기각: Gd와 TA 집단의 평균 SalePrice는 통계적으로 유의미한 차이가 있습니다.")
else:
    print(f"\np-value({test_result.pvalue:.6f}) >= 유의수준({alpha})")
    print("→ 귀무가설 기각 실패: 두 집단의 평균 SalePrice가 다르다고 볼 통계적 근거가 부족합니다.")

Gd 표본 수:  30
TA 표본 수:  30
Gd 평균:  190591.06666666668
TA 평균:  133586.66666666666

[Gd 정규성]
statistic:  0.934840752502985
p-value:  0.06610555406192758

[TA 정규성]
statistic:  0.966856779347431
p-value:  0.4571274954745825

[등분산성]
statistic:  6.062767675052423
p-value:  0.01679857021219461

선택한 검정 방법:  Welch t검정 (등분산 가정하지 않음)
검정통계량:  4.416993263639244
p-value:  5.4795626221121975e-05

p-value(0.000055) < 유의수준(0.05)
→ 귀무가설 기각: Gd와 TA 집단의 평균 SalePrice는 통계적으로 유의미한 차이가 있습니다.


### 필수 1 답변 작성란

- Q1. Shapiro-Wilk 검정의 귀무가설은 무엇인가요?  
<br> -표본이 나온 모집단의 분포가 정규분포다..
<br> - p-value가 0.05이하면 이 가설을 기가하고 정규성 위반 근거가 있다고 본다.

- Q2. Levene 검정의 귀무가설은 무엇인가요?  
<br> - 두 집단의 분산이 같다
<br> - 두 집단의 가격이 각 집단 평균 주변에 흩어진 정도가 같은지 확인한다.

- **Q3.** 가정 점검 결과 어떤 t검정 방법을 선택해야 하나요?  
<br> Welch t검정을 선택, 정규성 위반 근거는 없고
<br> Levene P-value가 0.017로 등분산성 위반 근거가 있다.

- **Q4.** 선택한 검정의 결과에 따르면 두 집단의 판매가격 차이는 유의한가요?
<br> 네. Welch t검정의 p-value가 약 0.05보다 작으므로 두 집단의 모집단 평균 판매가격이 같다는 귀무가설을 기각한다.

---

## 필수 2. 정규성과 등분산성이 모두 충족된 경우

### 문제 2-1. 난방 품질 `TA`와 `Fa` 집단의 판매가격 비교

#### 문제 설명

난방 품질이 `TA`(Typical/Average)인 주택과 `Fa`(Fair)인 주택의 판매가격을 비교하려고 합니다. 각 집단에서 20개씩 표본을 추출한 뒤 정규성과 등분산성을 확인하고 적절한 t검정 방법을 선택하세요.

#### 요구사항

1. `HeatingQC == "TA"`와 `HeatingQC == "Fa"`인 집단에서 `SalePrice`를 20개씩 추출하세요.
2. 두 집단의 표본 수와 평균을 확인하세요.
3. 두 집단이 독립집단인지 대응집단인지 판단하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 두 집단에 Levene 등분산 검정을 수행하세요.
6. 두 집단 모두 정규성을 충족하는지 확인하세요.
7. 등분산성 결과에 따라 독립표본 t검정 또는 Welch t검정 중 적절한 방법을 선택하세요.
8. 선택한 검정을 실행하고 검정통계량과 p-value를 출력하세요.
9. 두 집단의 판매가격 차이가 통계적으로 유의한지 해석하세요.

#### 해석 질문

**Q1.** 두 집단은 독립집단인가요, 대응집단인가요?  
**Q2.** 두 집단의 정규성 가정은 충족되나요?  
**Q3.** 두 집단의 등분산성 가정은 충족되나요?  
**Q4.** 가정 점검 결과 어떤 t검정 방법을 선택해야 하나요?  
**Q5.** 최종 검정 결과는 무엇을 의미하나요?

#### 제출 결과

- 집단별 표본 수와 평균
- 집단 관계 판단
- 정규성 및 등분산성 검정 결과
- t검정 방법과 선택 근거
- 검정통계량과 p-value
- 결과 해석
- Q1~Q5 답변


In [2]:
# 필수 2 코드를 작성하세요.
group_ta_heat = (
    df.loc[df["HeatingQC"]=="TA", "SalePrice"]
    .dropna()
    .sample(n=20, random_state=42))

group_fa_heat = (
 df.loc[df["HeatingQC"]=="Fa", "SalePrice"]
    .dropna()
    .sample(n=20, random_state=42))

print("TA 표본 수: ", len(group_ta_heat))
print("FA 표본 수: ", len(group_fa_heat))
print("TA 평균: ", group_ta_heat.mean())
print("FA 평균:", group_fa_heat.mean())

shapiro_ta_heat = stats.shapiro(group_ta_heat)
shapiro_fa_heat = stats.shapiro(group_fa_heat)
levene_heat = stats.levene(group_ta_heat, group_fa_heat)

print("\n[TA 정규성]")
print("statistic: ", shapiro_ta_heat.statistic)
print("p-value:", shapiro_ta_heat.pvalue)

print("\n[Fa 정규성]")
print("statistic: ", shapiro_fa_heat.statistic)
print("p-value:", shapiro_fa_heat.pvalue)

print("\n[등분산성]")
print("statistic: ", levene_heat.statistic)
print("p-value:", levene_heat.pvalue)

# 정규성과 등분산성이 모두 충족되므로 독립표본 t검정
test_heat = stats.ttest_ind(
    group_ta_heat,
    group_fa_heat
)

# t검정 결과 출력
print("\n[독립표본 t검정 결과]")
print("t-statistic: ", test_heat.statistic)
print("p-value: ", test_heat.pvalue)

# 유의수준(alpha) 설정 - 보통 0.05를 기준으로 씁니다
alpha = 0.05

if test_heat.pvalue < alpha:
    print(f"\np-value({test_heat.pvalue:.4f}) < 유의수준({alpha})")
    print("→ 귀무가설 기각: TA와 Fa 그룹의 평균 SalePrice는 통계적으로 유의미한 차이가 있습니다.")
else:
    print(f"\np-value({test_heat.pvalue:.4f}) >= 유의수준({alpha})")
    print("→ 귀무가설 기각 실패: 두 그룹의 평균 SalePrice가 다르다고 볼 통계적 근거가 부족합니다.")

# 효과크기(Cohen's d) 계산 - '통계적으로 유의미한 것'과 '실질적으로 큰 차이인 것'은 다르기 때문에 함께 확인합니다
mean_diff = group_ta_heat.mean() - group_fa_heat.mean()
pooled_std = ((group_ta_heat.std()**2 + group_fa_heat.std()**2) / 2) ** 0.5
cohens_d = mean_diff / pooled_std

print("\n[효과크기]")
print("평균 차이: ", mean_diff)
print("Cohen's d: ", cohens_d)

TA 표본 수:  20
FA 표본 수:  20
TA 평균:  130845.0
FA 평균: 122855.0

[TA 정규성]
statistic:  0.9736358683478321
p-value: 0.8290136550405987

[Fa 정규성]
statistic:  0.9319231671808381
p-value: 0.16814313675772813

[등분산성]
statistic:  1.310397035773581
p-value: 0.2594819416851383

[독립표본 t검정 결과]
t-statistic:  0.5583502977709472
p-value:  0.579880501062967

p-value(0.5799) >= 유의수준(0.05)
→ 귀무가설 기각 실패: 두 그룹의 평균 SalePrice가 다르다고 볼 통계적 근거가 부족합니다.

[효과크기]
평균 차이:  7990.0
Cohen's d:  0.17656586731894286


### 필수 2 답변 작성란

- **Q1.** 두 집단은 독립집단인가요, 대응집단인가요?  
<br> -> 필수 2번에서는 독립집단, 서로 다른 주택을 비교하여
<br> -> 같은 주택의 리모델링 전후처럼 관측값이 짝을 이루는 대응집단이 아님

- **Q2.** 두 집단의 정규성 가정은 충족되나요?  
<br> -> 두 집단 모두 Shapiro p-value가 0.05보다 크므로 정규성 위반 근거가 부족하다.

- **Q3.** 두 집단의 등분산성 가정은 충족되나요?  
<br> -> Levene p-value가 약 0.259이므로 0.05보다 크므로 등분산성 위반 근거가 부족하다.

- **Q4.** 가정 점검 결과 어떤 t검정 방법을 선택해야 하나요?  
<br> -> 실습의 선택 기준에 따라 등분산을 가정한 독립표본 t검정을 선택합니다.

- **Q5.** 최종 검정 결과는 무엇을 의미하나요?
<br> -> p-value가 약 0.58이므로 0.05보다 크다.
<br> -> 평균의 차이가 통계적으로 유의하지 않다.

=> 필수 2번에서 정규성과 등분산성의 위반 근거가 없으면 등분산을 가정한 독립표본 t검정을 선택한다.


---

## 과제. 난방 품질에 따른 t검정 방법 선택

### 문제 3-1. 난방 품질 `Ex`와 `TA` 집단의 판매가격 비교

#### 문제 설명

난방 품질이 `Ex`(Excellent)인 주택과 `TA`(Typical/Average)인 주택의 판매가격을 비교하려고 합니다. 각 집단에서 20개씩 표본을 추출한 뒤 필수 문제에서 학습한 **독립표본 t검정과 Welch t검정 선택 과정**을 독립적으로 적용하세요.

#### 요구사항

1. `HeatingQC == "Ex"`인 집단과 `HeatingQC == "TA"`인 집단에서 `SalePrice`를 20개씩 추출하세요.
2. 두 집단의 표본 수와 평균을 출력하세요.
3. 두 집단이 독립집단인지 대응집단인지 판단하세요.
4. 각 집단의 정규성과 두 집단의 등분산성을 검정하세요.
5. 두 집단 모두 정규성을 충족하는지 확인하세요.
6. 등분산성 결과에 따라 독립표본 t검정 또는 Welch t검정 중 적절한 방법을 선택하고 선택 이유를 작성하세요.
7. 선택한 검정을 실행하여 검정통계량과 p-value를 출력하세요.
8. 난방 품질에 따라 판매가격에 유의한 차이가 있는지 결론을 작성하세요.

#### 해석 질문

**Q1.** 두 집단의 정규성 가정은 충족되나요?  
**Q2.** 두 집단의 등분산성 가정은 충족되나요?  
**Q3.** 최종적으로 어떤 t검정 방법을 선택해야 하나요?  
**Q4.** 검정 결과 난방 품질에 따른 판매가격 차이는 통계적으로 유의한가요?

#### 제출 결과

- 표본 구성과 집단 관계 판단
- 정규성 및 등분산성 검정 결과
- 최종 t검정 선택과 근거
- 검정통계량과 p-value
- 결과 해석
- Q1~Q4 답변


In [ ]:
# 과제 코드를 작성하세요.

### 과제 답변 작성란

- **Q1.**
- **Q2.**
- **Q3.**
- **Q4.**

---

## 실습 마무리

1. 두 집단을 비교하기 전에 어떤 데이터 특성을 먼저 확인해야 하나요?
<br> -> 비교할 값은 판매가격처럼 수치형 변수인지, 범주형 변수인지 두 집단은 독립, 대응인지 확인해야 한다.

2. 정규성 검정과 등분산 검정에서 `p > 0.05`는 무엇을 의미하나요?
<br> -> Shapiro 검정은 정규성, Levene 검정은 등분산성, 마지막 t검정은 두 모집단의 평균 차이를 확인한다.

3. 두 집단 모두 정규성을 충족하고 등분산성도 충족하면 어떤 검정을 사용할 수 있나요?
<br> -> 정규성, 등분산성 검정에서 p > 0.05이면 해당 가정의 위반 근거가 부족하다라는 뜻
<br> -> 가정이 참이라고 증명된 것은 아니다.
<br> -> 독립표본 t검정을 선택한다.

4. 두 집단 모두 정규성을 충족하지만 등분산성이 위반되면 어떤 검정을 사용할 수 있나요?
<br> -> Welch t검정을 선택한다.

5. 독립표본 t검정과 Welch t검정을 선택할 때 정규성과 등분산성을 함께 확인해야 하는 이유는 무엇인가요?
<br> -> 정규성 평균에 퍼짐 정도 일반적으로 예상이 가능한 범주
<br> -> 등분산성 두 집단 사이 얼마나 유사한 지